In [1]:
import warnings
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
warnings.filterwarnings('ignore', category=FutureWarning)


# загрухка данных 
df = pd.read_csv("train.csv")
# вывод первые 5 строк 
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [2]:
print("Количество пропущенных значений в каждом столбце:")
print(df.isnull().sum())

# вывод только столбцов, где есть пропуски
print("\nСтолбцы с пропусками:")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

Количество пропущенных значений в каждом столбце:
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Столбцы с пропусками:
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
dtype: int64


In [3]:
# удаление столбец Name
df.drop('Name', axis='columns', inplace=True)

# заполняем категориальные признаки модой
df['HomePlanet'] = df['HomePlanet'].fillna(df['HomePlanet'].mode()[0])
df['CryoSleep'] = df['CryoSleep'].fillna(df['CryoSleep'].mode()[0])
df['Cabin'] = df['Cabin'].fillna(df['Cabin'].mode()[0])
df['Destination'] = df['Destination'].fillna(df['Destination'].mode()[0])
df['VIP'] = df['VIP'].fillna(df['VIP'].mode()[0])

# заполняем числовые признаки медианой
df['Age'] = df['Age'].fillna(df['Age'].median())
df['RoomService'] = df['RoomService'].fillna(df['RoomService'].median())
df['FoodCourt'] = df['FoodCourt'].fillna(df['FoodCourt'].median())
df['ShoppingMall'] = df['ShoppingMall'].fillna(df['ShoppingMall'].median())
df['Spa'] = df['Spa'].fillna(df['Spa'].median())
df['VRDeck'] = df['VRDeck'].fillna(df['VRDeck'].median())

print("Количество пропусков после заполнения:")
print(df.isnull().sum())

total_nulls = df.isnull().sum().sum()
print(f"\nОбщее количество пропусков: {total_nulls}")

if total_nulls == 0:
    print("пропуски заполнены")
else:
    print("есть незаполненные пропуски")

Количество пропусков после заполнения:
PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
dtype: int64

Общее количество пропусков: 0
пропуски заполнены


In [4]:
# объект нормализатора
scaler = MinMaxScaler()

# список числовых столбцов, которые нужно нормализовать
numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for col in numeric_cols:
    df[col] = scaler.fit_transform(df[[col]])

print("Первые 5 строк нормализованных числовых столбцов:")
print(df[numeric_cols].head())

print("\nМинимальные значения после нормализации (должны быть 0.0):")
print(df[numeric_cols].min())

print("\nМаксимальные значения после нормализации (должны быть 1.0):")
print(df[numeric_cols].max())

Первые 5 строк нормализованных числовых столбцов:
        Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck
0  0.493671     0.000000   0.000000      0.000000  0.000000  0.000000
1  0.303797     0.007608   0.000302      0.001064  0.024500  0.001823
2  0.734177     0.003001   0.119948      0.000000  0.299670  0.002030
3  0.417722     0.000000   0.043035      0.015793  0.148563  0.007997
4  0.202532     0.021149   0.002348      0.006428  0.025214  0.000083

Минимальные значения после нормализации (должны быть 0.0):
Age             0.0
RoomService     0.0
FoodCourt       0.0
ShoppingMall    0.0
Spa             0.0
VRDeck          0.0
dtype: float64

Максимальные значения после нормализации (должны быть 1.0):
Age             1.0
RoomService     1.0
FoodCourt       1.0
ShoppingMall    1.0
Spa             1.0
VRDeck          1.0
dtype: float64


In [6]:
# список категориальных столбцов для кодирования
categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Transported 
df_encoded['Transported'] = df_encoded['Transported'].astype(int)

# удаляем столбцы Name и Cabin — они не нужны для модели
df_encoded.drop(['Name', 'Cabin'], axis='columns', inplace=True)
df_clean = df_encoded.drop('PassengerId', axis='columns')

bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("Размер датасета после кодирования (строки, столбцы):", df_encoded.shape)

print("\nПример преобразования (столбцы HomePlanet):")
print(df_encoded.filter(like='HomePlanet').head())

print("\nЦелевая переменная Transported (0 = нет, 1 = да):")
print(df_encoded['Transported'].head())

print("\nВсе столбцы датасета:")
print(df_encoded.columns.tolist())

print("\nТипы данных всех столбцов:")
print(df_encoded.dtypes)

KeyError: "['Name'] not found in axis"

In [7]:
# Сохраняем датасет в новый CSV-файл
# index=False убирает лишний столбец с номерами строк
df_encoded.to_csv("processed_spaceship_titanic.csv", index=False)
print("Предобработанный датасет успешно сохранен!")

Предобработанный датасет успешно сохранен!
